# Chat Interactions - Sample Outputs & Overview

This notebook loads the `chat_interactions.jsonl` log file and provides a high-level overview of what's being captured: field structure, value distributions, timing metrics, and sample records.

In [ ]:
import json
import pandas as pd
from pathlib import Path
from datetime import datetime

LOG_PATH = Path("../logs/chat_interactions.jsonl")

# Load all interactions
records = []
with open(LOG_PATH) as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

df = pd.DataFrame(records)
df["timestamp"] = pd.to_datetime(df["timestamp"])
print(f"Loaded {len(df)} interactions")

## Schema & Field Overview
What fields are captured in each log entry?

In [ ]:
print("Columns:", list(df.columns))
print()
df.dtypes

In [ ]:
# Basic stats
print(f"Date range: {df['timestamp'].min()} to {df['timestamp'].max()}")
print(f"Unique sessions: {df['session_id'].nunique()}")
print(f"Total interactions: {len(df)}")
print(f"Avg questions per session: {len(df) / df['session_id'].nunique():.1f}")

## Sample Records
Look at a few raw records to see what each interaction captures.

In [ ]:
# Show a random sample of questions and their metadata (excluding full answer text)
sample = df.sample(min(5, len(df)), random_state=42)
sample[["timestamp", "session_id", "question", "intent", "num_sources",
        "retrieval_time_seconds", "answer_time_seconds", "total_time_seconds"]]

In [ ]:
# View one complete record with full answer
row = df.iloc[0]
print(f"Question: {row['question']}")
print(f"Intent:   {row['intent']}")
print(f"Sources:  {row['num_sources']}")
print(f"Times:    retrieval={row['retrieval_time_seconds']:.2f}s, "
      f"answer={row['answer_time_seconds']:.2f}s, "
      f"total={row['total_time_seconds']:.2f}s")
print(f"\nAnswer:\n{row['answer'][:500]}...")

## Intent Distribution
How are queries classified across intent categories?

In [ ]:
intent_counts = df["intent"].value_counts()
print(intent_counts)
print(f"\nAs percentages:")
print((intent_counts / len(df) * 100).round(1).astype(str) + "%")

## Timing Metrics
How long do retrieval, answer generation, and total processing take?

In [ ]:
timing_cols = ["retrieval_time_seconds", "answer_time_seconds", "total_time_seconds"]
df[timing_cols].describe().round(2)

In [ ]:
# Timing breakdown by intent
df.groupby("intent")[timing_cols].mean().round(2)

## Source Analysis
What sources are being retrieved most frequently?

In [ ]:
# Flatten all sources across interactions
all_sources = []
for _, row in df.iterrows():
    for src in row["sources"]:
        all_sources.append({
            "parent_id": src.get("parent_id"),
            "title": src.get("title"),
            "collection": src.get("collection"),
        })

sources_df = pd.DataFrame(all_sources)
print(f"Total source citations: {len(sources_df)}")
print(f"Unique documents cited: {sources_df['parent_id'].nunique()}")
print()
print("Most cited documents:")
sources_df.groupby(["parent_id", "title"]).size().sort_values(ascending=False).head(10)

In [ ]:
# Collection distribution
print("Citations by collection:")
sources_df["collection"].value_counts()

## Question Length & Answer Length

In [ ]:
df["question_length"] = df["question"].str.len()
df["answer_length"] = df["answer"].str.len()
df["question_word_count"] = df["question"].str.split().str.len()
df["answer_word_count"] = df["answer"].str.split().str.len()

print("Question length (chars):")
print(df["question_length"].describe().round(0))
print()
print("Answer length (words):")
print(df["answer_word_count"].describe().round(0))

## Activity Over Time

In [ ]:
# Interactions per day
daily = df.set_index("timestamp").resample("D").size()
daily = daily[daily > 0]
print("Interactions per day:")
for date, count in daily.items():
    print(f"  {date.strftime('%Y-%m-%d')}: {count}")

In [ ]:
# Interactions per hour of day
df["hour"] = df["timestamp"].dt.hour
hourly = df["hour"].value_counts().sort_index()
print("Interactions by hour (UTC):")
for hour, count in hourly.items():
    print(f"  {hour:02d}:00 - {count}")